# Pipeline Cancellation

## What you'll learn

- Cancel a running pipeline programmatically with `pipeline.cancel()`
- Inspect authoritative cancellation state and acknowledgements
- Re-run a cancelled pipeline — succeeded steps are cached, cancelled steps re-execute
- How Ctrl+C / SIGTERM signal handling works automatically

**Prerequisites:** [Resume and Caching](../03-caching/01-resume-and-caching.ipynb), [Error Handling in Practice](02-error-visibility.ipynb)  
**Estimated time:** 10 minutes  
**GPU required:** No.

---

In production pipelines, you sometimes need to stop early — bad upstream
data, wrong parameters, or resource limits. The framework supports
runner-managed cancellation: call `pipeline.cancel()` (or press Ctrl+C) to
request that active work stops and prevent later steps from dispatching.

| Section | What you'll see |
|---------|----------------|
| **Programmatic cancellation** | `pipeline.cancel()` stops remaining steps |
| **Inspecting cancelled results** | `status` and `cancellation_status` carry lifecycle facts |
| **Re-running after cancellation** | Succeeded steps are cached; cancelled steps re-execute |
| **Signal handling and scripts** | Ctrl+C and provider cancellation behavior |

In [ ]:
from __future__ import annotations

from artisan.operations.examples import Wait
from artisan.orchestration import PipelineManager, StepDisposition, StepStatus
from artisan.utils import tutorial_setup
from artisan.visualization import inspect_pipeline

In [ ]:
env = tutorial_setup("pipeline_cancellation")

## Programmatic cancellation

Build a 4-step pipeline using `Wait` (a lightweight operation that
sleeps for a configurable duration). Run the first two steps, call
`pipeline.cancel()`, then run the remaining two. Those new attempts
record confirmed cancellation without dispatching execution work.

In [ ]:
pipeline = PipelineManager.create(
    name="cancellation_demo",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)

step_0 = pipeline.run(
    operation=Wait,
    name="wait_0",
    params={"duration": 0.1},
)
step_1 = pipeline.run(
    operation=Wait,
    name="wait_1",
    params={"duration": 0.11},
)
print(f"Step 0: {step_0.succeeded_count} succeeded")
print(f"Step 1: {step_1.succeeded_count} succeeded")

Steps 0–1 completed normally. Now cancel the pipeline and run
two more steps:

In [ ]:
pipeline.cancel()

step_2 = pipeline.run(
    operation=Wait,
    name="wait_2",
    params={"duration": 0.12},
)
step_3 = pipeline.run(
    operation=Wait,
    name="wait_3",
    params={"duration": 0.13},
)

result = pipeline.finalize()
print(f"Pipeline: {result['total_steps']} steps, success={result['overall_success']}")
assert step_2.status is StepStatus.CANCELLED
assert step_3.status is StepStatus.CANCELLED
assert not result["overall_success"]

Steps 2–3 became `cancelled` without dispatching an execution. Their
histories contain `pending`, cancellation `requested`, cancellation
`confirmed`, and the terminal `cancelled` snapshot.

## Inspecting cancelled results

Lifecycle facts are typed fields, not metadata flags:

| Field | Value | Meaning |
|-------|-------|---------|
| `status` | `StepStatus.CANCELLED` | The attempt reached the cancelled terminal state |
| `cancellation_status` | `confirmed` | The cancellation was acknowledged |

A request alone is not enough to fabricate a cancelled result. Rejected work
finishes naturally, while an indeterminate acknowledgement fails closed.

In [ ]:
for step in [step_0, step_1, step_2, step_3]:
    if step.status is StepStatus.CANCELLED:
        print(
            f"Step {step.step_number} ({step.step_name}): "
            f"cancelled — {step.cancellation_status.value}"
        )
    else:
        print(
            f"Step {step.step_number} ({step.step_name}): "
            f"{step.succeeded_count}/{step.total_count} succeeded"
        )

Cancelled steps have zero counts and expose no outputs. They are not
successful steps: the pipeline summary's `overall_success` is `False`.
The acknowledgement distinguishes a confirmed stop from an unknown outcome.

In [ ]:
inspect_pipeline(env.delta_root)

`inspect_pipeline` shows every accepted attempt's current snapshot from
the steps table. Steps 2–3 therefore appear with `status = "cancelled"`
even though no execution work was dispatched.

:::{note}
The `inspect_pipeline` view shows what was persisted to Delta Lake.
For the typed acknowledgement and item counts, inspect the in-memory
`StepResult` values directly as shown above.
:::

## Re-running after cancellation

Cancelled steps do not supply whole-step cache hits. In this example they never
dispatched any work, so there are no execution units to recover. Each Wait step
uses a different duration to give it a distinct computation: steps 0–1 load
from cache, while steps 2–3 execute fresh.

In [ ]:
env = tutorial_setup("pipeline_cancellation", clean=False)

pipeline2 = PipelineManager.create(
    name="cancellation_rerun",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)

In [ ]:
step_0r = pipeline2.run(
    operation=Wait,
    name="wait_0",
    params={"duration": 0.1},
)
step_1r = pipeline2.run(
    operation=Wait,
    name="wait_1",
    params={"duration": 0.11},
)
step_2r = pipeline2.run(
    operation=Wait,
    name="wait_2",
    params={"duration": 0.12},
)
step_3r = pipeline2.run(
    operation=Wait,
    name="wait_3",
    params={"duration": 0.13},
)

result2 = pipeline2.finalize()
print(f"Pipeline: {result2['total_steps']} steps, success={result2['overall_success']}")
assert step_0r.disposition is StepDisposition.CACHE_HIT
assert step_1r.disposition is StepDisposition.CACHE_HIT
assert step_2r.disposition is StepDisposition.EXECUTED
assert step_3r.disposition is StepDisposition.EXECUTED
assert result2["overall_success"]

Log output shows "CACHED" for steps 0–1 and normal execution for
steps 2–3. See [Resume and Caching](../03-caching/01-resume-and-caching.ipynb)
for more on caching.

In [ ]:
for step in [step_0r, step_1r, step_2r, step_3r]:
    print(
        f"Step {step.step_number} ({step.step_name}): "
        f"{step.status.value}, {step.disposition.value}: "
        f"{step.succeeded_count}/{step.total_count} succeeded"
    )

In [ ]:
inspect_pipeline(env.delta_root)

(recover-finished-units)=
## Recover finished units

A cancelled step may already have finished some independent execution units.
Their staging is retained. On the next pipeline start, recovery validates and
commits those completed units before cache lookup.

Use the companion `PausedTransformer`, a `DataTransformer` subclass that lets
one dataset finish and holds the other until a release file exists. This gives
us a repeatable cancellation point. It also records each input's actual
invocations so we can prove that recovered work did not execute twice. The
release file changes timing only; both inputs use the normal transformation.

This example creates a separate tutorial directory and cleans its previous
contents. `preserve_staging=True` keeps evidence even after successful recovery.

In [ ]:
import time
from pathlib import Path

from staging_recovery_demo import PausedTransformer

from artisan.operations.examples import DataGenerator
from artisan.visualization import inspect_step

recovery_env = tutorial_setup("staging_recovery")
demo_root = Path(recovery_env.runs_dir).resolve()
interrupted = PipelineManager.create(
    name="interrupted_transform",
    delta_root=recovery_env.delta_root,
    staging_root=recovery_env.staging_root,
    working_root=recovery_env.working_root,
    preserve_staging=True,
)
source = interrupted.run(DataGenerator, params={"count": 2, "seed": 42})
transform_params = {"demo_root": str(demo_root), "noise_amplitude": 0.0}
future = interrupted.submit(
    PausedTransformer,
    inputs={"dataset": source.output("datasets")},
    params=transform_params,
    batch_strategy={"artifacts_per_unit": 1, "units_per_worker": 1, "max_workers": 2},
)

Wait for the first worker's completion seal, then request cancellation while
the second input is paused. This waits for actual finished staging rather than
assuming that a particular sleep was long enough.

In [ ]:
staged_step = Path(recovery_env.staging_root) / (
    f"{future.step_number}_{PausedTransformer.name}"
)
deadline = time.monotonic() + 30
try:
    while not (seals := list(staged_step.rglob("executions.parquet"))):
        if future.done or time.monotonic() > deadline:
            msg = "The first tutorial execution did not finish"
            raise RuntimeError(msg)
        time.sleep(0.02)
    finished_seal = seals[0]
    retained_bytes = {
        path: path.read_bytes() for path in finished_seal.parent.glob("*.parquet")
    }
finally:
    interrupted.cancel()
    interrupted.finalize()

cancelled = future.result()
assert cancelled.status is StepStatus.CANCELLED
assert all(path.read_bytes() == content for path, content in retained_bytes.items())
assert inspect_step(
    recovery_env.delta_root,
    cancelled.step_number,
    pipeline_run_id=interrupted.config.pipeline_run_id,
).is_empty()

The old driver has stopped and the cancelled step has no accepted outputs.
Start a new manager on these roots. Its default recovery runs before we submit
anything. Release the paused input and rerun the same computation.

In [ ]:
(demo_root / "release").touch()
recovered = PipelineManager.create(
    name="recovered_transform",
    delta_root=recovery_env.delta_root,
    staging_root=recovery_env.staging_root,
    working_root=recovery_env.working_root,
    preserve_staging=True,
)
source_again = recovered.run(DataGenerator, params={"count": 2, "seed": 42})
completed = recovered.run(
    PausedTransformer,
    inputs={"dataset": source_again.output("datasets")},
    params=transform_params,
    batch_strategy={"artifacts_per_unit": 1, "units_per_worker": 1, "max_workers": 2},
)
recovered.finalize()

assert completed.status is StepStatus.SUCCEEDED
assert (demo_root / "dataset_00000.calls").read_text().splitlines() == ["executed"]
assert (
    inspect_step(
        recovery_env.delta_root,
        completed.step_number,
        pipeline_run_id=recovered.config.pipeline_run_id,
    ).height
    == 2
)
assert all(path.read_bytes() == content for path, content in retained_bytes.items())

In [ ]:
old_history = inspect_pipeline(
    recovery_env.delta_root, pipeline_run_id=interrupted.config.pipeline_run_id
)
assert (
    old_history.filter(old_history["step"] == cancelled.step_number)["status"].item()
    == "cancelled"
)
assert inspect_step(
    recovery_env.delta_root,
    cancelled.step_number,
    pipeline_run_id=interrupted.config.pipeline_run_id,
).is_empty()
inspect_pipeline(
    recovery_env.delta_root, pipeline_run_id=recovered.config.pipeline_run_id
)

The first input executed once. Recovery made its original completed execution
available for reuse; the new run now exposes both outputs. The original step
still reports `cancelled` and exposes none.

Uncommitted files are retained even with `preserve_staging=False`. That default
allows cleanup only after verified commitment. Set `recover_staging=False` to
leave earlier staging untouched at startup. See
[Staging preservation and recovery](../../concepts/storage-and-delta-lake.md#staging-preservation)
for incomplete work, conflicting evidence, and the single-writer requirement.

## Signal handling and interactive cancellation

Scripts can request cancellation with Ctrl+C (SIGINT) or SIGTERM. Artisan
installs handlers when the first step starts and restores the previous handlers
in `finalize()`. In notebook or background-thread code, use `cancel()` directly.

The first signal requests cancellation. A second restores the **previous signal
handlers**. Subsequent signals follow those handlers; in a typical Python
terminal, another Ctrl+C raises `KeyboardInterrupt`. This escalation does not
guarantee that external tools have completed or undone their side effects.

:::{note}
Cancellation reaches the active runner while execution is in progress. The local
runner allows a brief cooperative grace period, then can terminate its owned
worker processes. An in-flight execute phase is not guaranteed to finish.

Artisan exposes outputs according to the confirmed lifecycle outcome. A cancelled
attempt exposes no accepted outputs, while finished execution evidence is retained
for recovery and diagnosis. This does not guarantee cleanup of arbitrary files or external
side effects produced by a tool.
:::

To try interactive cancellation, run the companion script from a
terminal:

```bash
pixi run python docs/tutorials/05-errors-and-control/cancel_demo.py
```

The script submits 10 Wait steps that each sleep for 30 seconds.
Press Ctrl+C while it's running — you'll see succeeded and cancelled
steps in the summary. Press Ctrl+C
again to restore the previous signal handlers if shutdown takes too long.

## Runner-provider cancellation

Artisan forwards pipeline cancellation to the active runner's lifecycle
router. Optional runner providers own cancellation of their exact external
jobs or processes and return a typed acknowledgement. `confirmed` permits a
`cancelled` terminal state, `rejected` lets work finish naturally, and `unknown`
fails the attempt closed. Consult the provider's documentation for
scheduler-specific behavior.

## Summary

You cancelled a pipeline, inspected confirmed cancellation, and reran it. Successful
steps were reusable. Finished units from a cancelled step were recovered without
changing its history; unfinished work executed in the new run. Check `status` and
`cancellation_status` before interpreting an outcome: a cancellation request alone
does not prove that work stopped.

The active runner controls how work stops. The local runner can terminate owned
workers after its grace period. Use `inspect_pipeline` for persisted step state
and `StepResult` for the typed acknowledgement.

## Next steps

- [Resume and Caching](../03-caching/01-resume-and-caching.ipynb) — Reuse successful work
- [Error Handling in Practice](02-error-visibility.ipynb) — Read failure evidence
- [Error Handling](../../concepts/error-handling.md) — Failure and cancellation semantics
- [Configure Execution](../../how-to-guides/configuring-execution.md) — Choose a runner